In [2]:
import pandas as pd
import torch
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import plotly.express as px
import json

In [1]:
# connect google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
newsdata_df = pd.read_csv("/content/drive/MyDrive/News_Analysis_Data/Thesis_Datasets_GH/newsdata_df_13257.csv")

In [4]:
newsdata_df.head()

,article_id,title,link,keywords,creator,video_url,description,pubDate,pubDateTZ,image_url,source_id,source_name,source_url,source_icon,language,country,category
0,26295da91d37241df96204e9baae132c,CMHO कार्यालय का अतिरिक्त कार्यभार बना चर्चा म...,https://www.bhaskar.com/local/rajasthan/sawai-...,NaN,NaN,NaN,अपनों पर सितम गैरों पर रहम की कहानी सवाई माधोप...,2025-02-13 09:21:00,UTC,https://images.bhaskarassets.com/thumb/1000x10...,bhaskar_hindi,Bhaskar,https://www.bhaskar.com,https://i.bytvi.com/domain_icons/bhaskar_hindi...,hindi,['india'],"['domestic', 'other']"
1,0b67e6100b1fcfb8e1a1196fb4ab01b7,Crude Oil Prices Decline as US-Russia Talks Ra...,https://in.investing.com/analysis/crude-oil-pr...,NaN,['ING Economic and Financial Analysis'],NaN,NaN,2025-02-13 09:21:00,UTC,NaN,investing_in,Investing India,https://in.investing.com,NaN,english,['india'],['business']
2,38587c228f49d1b95978e6f531a6cd2b,বিয়া চলি থকাৰ সময়তেই উপস্থিত আৰক্ষী; বিবাহথল...,https://www.assamtv9.com/assam/groom-and-bride...,['assam news'],['Raj Saikia'],NaN,আলহী-অতিথিৰ সৈতে ৰজনজনাই থকা বিয়াঘৰত হঠাৎ উপস...,2025-02-13 09:21:00,UTC,https://images.assamtv9.com/uploads/2025/02/Ch...,assamtv9,Assamese News Today,https://www.assamtv9.com,https://i.bytvi.com/domain_icons/assamtv9.png,assamese,['india'],['top']
3,cb53bbc7258b834c1342ac7a98f0390a,Aero India 2025: ಪ್ರಮುಖ ನೌಕಾ ವಾಯುಯಾನ ತಂತ್ರಜ್ಞಾ...,https://www.kannadaprabha.com/karnataka/2025/F...,NaN,['Sumana Upadhyaya'],NaN,NaN,2025-02-13 09:20:54,UTC,https://media.assettype.com/kannadaprabha%2F20...,kannadaprabha,Kannada Prabha Online,https://www.kannadaprabha.com,NaN,kannada,['india'],['top']
4,b71d1cb1a8db3d55cd2d6410659b9ce9,Health experts and enthusiasts react to OTT's ...,https://timesofindia.indiatimes.com/life-style...,NaN,NaN,NaN,"The Netflix show Apple Cider Vinegar, starring...",2025-02-13 09:20:54,UTC,https://static.toiimg.com/thumb/msid-118209993...,toi,The Times Of India,https://timesofindia.indiatimes.com,https://i.bytvi.com/domain_icons/toi.png,english,['india'],['top']


In [ ]:
newsdata_df.value_counts("language")

,count
language,
english,6243
hindi,3777
malayalam,774
tamil,721
telugu,430
bengali,339
kannada,279
gujarati,240
marathi,215


In [5]:
# check the data range of the articles from pubDate column
newsdata_df["pubDate"].min(), newsdata_df["pubDate"].max()

('2025-02-13 09:12:56', '2025-02-21 19:44:03')

### Load the IndicSBERT -> Generate Embeddings for articles' `title` for dataset from NewsData.io.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load the L3Cube-IndicSBERT model
model = SentenceTransformer("l3cube-pune/indic-sentence-bert-nli")
model = model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
title_embeddings = []
with tqdm(total=len(newsdata_df), desc="Generating Title Embeddings", dynamic_ncols=True, leave=True) as pbar:
    for title in newsdata_df["title"].astype(str):
        embedding = model.encode(title, convert_to_tensor=True, device=device).cpu().numpy()  
        title_embeddings.append(embedding.tolist()) 
        pbar.update(1)

# Add embeddings to DataFrame
newsdata_df["title_embeddings"] = title_embeddings

Generating Title Embeddings: 100%|██████████| 13257/13257 [03:59<00:00, 55.40it/s]


In [ ]:
newsdata_df.head()

,article_id,title,link,keywords,creator,video_url,description,pubDate,pubDateTZ,image_url,source_id,source_name,source_url,source_icon,language,country,category,title_embeddings
0,26295da91d37241df96204e9baae132c,CMHO कार्यालय का अतिरिक्त कार्यभार बना चर्चा म...,https://www.bhaskar.com/local/rajasthan/sawai-...,NaN,NaN,NaN,अपनों पर सितम गैरों पर रहम की कहानी सवाई माधोप...,2025-02-13 09:21:00,UTC,https://images.bhaskarassets.com/thumb/1000x10...,bhaskar_hindi,Bhaskar,https://www.bhaskar.com,https://i.bytvi.com/domain_icons/bhaskar_hindi...,hindi,['india'],"['domestic', 'other']","[0.0019764662720263004, 0.0009495333069935441,..."
1,0b67e6100b1fcfb8e1a1196fb4ab01b7,Crude Oil Prices Decline as US-Russia Talks Ra...,https://in.investing.com/analysis/crude-oil-pr...,NaN,['ING Economic and Financial Analysis'],NaN,NaN,2025-02-13 09:21:00,UTC,NaN,investing_in,Investing India,https://in.investing.com,NaN,english,['india'],['business'],"[-0.002035742159932852, 0.00711246021091938, 0..."
2,38587c228f49d1b95978e6f531a6cd2b,বিয়া চলি থকাৰ সময়তেই উপস্থিত আৰক্ষী; বিবাহথল...,https://www.assamtv9.com/assam/groom-and-bride...,['assam news'],['Raj Saikia'],NaN,আলহী-অতিথিৰ সৈতে ৰজনজনাই থকা বিয়াঘৰত হঠাৎ উপস...,2025-02-13 09:21:00,UTC,https://images.assamtv9.com/uploads/2025/02/Ch...,assamtv9,Assamese News Today,https://www.assamtv9.com,https://i.bytvi.com/domain_icons/assamtv9.png,assamese,['india'],['top'],"[0.021987082436680794, 0.004995401483029127, -..."
3,cb53bbc7258b834c1342ac7a98f0390a,Aero India 2025: ಪ್ರಮುಖ ನೌಕಾ ವಾಯುಯಾನ ತಂತ್ರಜ್ಞಾ...,https://www.kannadaprabha.com/karnataka/2025/F...,NaN,['Sumana Upadhyaya'],NaN,NaN,2025-02-13 09:20:54,UTC,https://media.assettype.com/kannadaprabha%2F20...,kannadaprabha,Kannada Prabha Online,https://www.kannadaprabha.com,NaN,kannada,['india'],['top'],"[-0.0076497276313602924, 0.006926835980266333,..."
4,b71d1cb1a8db3d55cd2d6410659b9ce9,Health experts and enthusiasts react to OTT's ...,https://timesofindia.indiatimes.com/life-style...,NaN,NaN,NaN,"The Netflix show Apple Cider Vinegar, starring...",2025-02-13 09:20:54,UTC,https://static.toiimg.com/thumb/msid-118209993...,toi,The Times Of India,https://timesofindia.indiatimes.com,https://i.bytvi.com/domain_icons/toi.png,english,['india'],['top'],"[0.012143010273575783, 0.018910318613052368, 0..."


In [ ]:
# remove rows where language is vietnamese, khmer or swahili
newsdata_df = newsdata_df[~newsdata_df["language"].isin(["vietnamese", "khmer", "swahili"])]

In [ ]:
embedding_matrix = np.array(newsdata_df["title_embeddings"].tolist())

In [ ]:
tsne = TSNE(
    n_components=2,
    perplexity=50,
    max_iter=2000,
    learning_rate=200,
    metric="cosine",
    random_state=42
)

tsne_result = tsne.fit_transform(embedding_matrix)


/usr/local/lib/python3.11/dist-packages/sklearn/manifold/_t_sne.py:1164: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


In [ ]:
# creating new DataFrame with selected columns and x/y
tsne_df = newsdata_df[["title", "source_name", "pubDate", "language"]].copy()
tsne_df["x"] = tsne_result[:, 0]
tsne_df["y"] = tsne_result[:, 1]

In [ ]:
# Plot t-SNE
fig = px.scatter(
    tsne_df, x="x", y="y", color="language",
    hover_data=["title", "language", "source_name"],
    title="t-SNE Clustering of News Articles"
)
fig.show()